# Intelligent Revenue Cycle Management Agent

In [1]:
!pip install -U fastapi==0.118.1 uvicorn websockets==10.4 gradio==4.44.0 langchain langchain-google-genai google-generativeai langgraph pydantic python-dotenv --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.9/84.9 kB 1.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.19.0 requires websockets<16.0.0,>=15.0.1, but you have websockets 10.4 which is incompatible.
google-genai 1.52.0 requires websockets<15.1.0,>=13.0.0, but you have websockets 10.4 which is incompatible.
dataproc-spark-connect 0.8.3 requires websockets>=14.0, but you have websockets 10.4 which is incompatible.
yfinance 0.2.66 requires websockets>=13.0, but you have websockets 10.4 which is incompatible.


# 1) Imports & Configs

In [2]:
import os
import time
import uuid
import logging
from typing import Dict, Any, Optional, List

from fastapi import FastAPI
from pydantic import BaseModel
import gradio as gr
from dotenv import load_dotenv

# Load environment variables from .env if present
load_dotenv()  # sets os.environ["GOOGLE_API_KEY"] if .env exists

# LangChain + Gemini
try:
    from langchain_google_genai import ChatGoogleGenerativeAI
    from langchain.prompts import PromptTemplate
    from langchain.chains import LLMChain
    HAS_GEMINI = True
except ImportError:
    HAS_GEMINI = False

# LangGraph
try:
    from langgraph.graph import StateGraph, END
    HAS_LANGGRAPH = True
except ImportError:
    HAS_LANGGRAPH = False

logging.basicConfig(level=logging.INFO)
print("✅ Imports complete. Gemini available:", HAS_GEMINI, " LangGraph available:", HAS_LANGGRAPH)

✅ Imports complete. Gemini available: False  LangGraph available: True


In [3]:
class Tool:
    name: str = "tool"
    description: str = "Generic tool"

    def __call__(self, *args, **kwargs):
        return self.call(*args, **kwargs)

    def call(self, *args, **kwargs):
        raise NotImplementedError


class InMemorySessionService:
    def __init__(self):
        self.sessions: Dict[str, Dict[str, Any]] = {}

    def create_session(self, initial_state: Dict[str, Any]) -> str:
        session_id = str(uuid.uuid4())
        self.sessions[session_id] = initial_state
        logging.info(f"Created session {session_id}")
        return session_id

    def get_state(self, session_id: str) -> Optional[Dict[str, Any]]:
        return self.sessions.get(session_id)

    def update_state(self, session_id: str, new_state: Dict[str, Any]):
        self.sessions[session_id] = new_state
        logging.info(f"Updated session {session_id}")


session_service = InMemorySessionService()
long_term_memory: List[Dict[str, Any]] = []  # Memory bank


# 2) Tools

In [4]:
class InsuranceCheckTool(Tool):
    name = "insurance_check"
    description = "Verify if an insurance number is valid (mock rule: AET* or BLU*)."

    def call(self, insurance_number: str) -> bool:
        print("InsuranceCheckTool.call")
        # Very simple rule: AET* or BLU* are "valid"
        return insurance_number.startswith("AET") or insurance_number.startswith("BLU")


class ClaimSubmitTool(Tool):
    name = "claim_submit"
    description = "Submit a claim (simulated), returns status, claim_id or reason."

    def call(self, codes: List[str], verified: bool) -> Dict[str, Any]:
        print("ClaimSubmitTool.call")
        # Simulate an external API call / long-running operation
        time.sleep(1.0)  # simulate latency
        if not verified:
            return {"status": "Rejected", "reason": "Insurance not verified"}
        if not codes:
            return {"status": "Rejected", "reason": "Missing codes"}
        return {"status": "Submitted", "claim_id": str(uuid.uuid4())}


class MockSearchTool(Tool):
    name = "mock_search"
    description = "Mock search tool that returns fake search results for a query."

    def call(self, query: str) -> str:
        print("MockSearchTool.call")
        # In real system: call Google Search / medical DB.
        return f"[Mock search results for: '{query}']"


class CodeExecutionTool(Tool):
    """Toy code execution / calculator tool."""
    name = "code_exec"
    description = "Safe calculator that evaluates basic math expressions."

    def call(self, expression: str) -> str:
        print("CodeExecutionTool.call")
        try:
            # VERY restricted eval: integers + operators only (demo only)
            allowed_chars = "0123456789+-*/(). "
            if any(ch not in allowed_chars for ch in expression):
                return "Error: disallowed characters in expression."
            value = eval(expression, {"__builtins__": {}})
            return str(value)
        except Exception as e:
            return f"Error evaluating expression: {e}"


insurance_tool = InsuranceCheckTool()
claim_tool = ClaimSubmitTool()
search_tool = MockSearchTool()
code_tool = CodeExecutionTool()


class ToolRegistry:

    def __init__(self):
        self.tools: Dict[str, Dict[str, Any]] = {}

    def register(self, tool: Tool):
        self.tools[tool.name] = {"instance": tool, "description": getattr(tool, "description", "")}

    def list_tools(self) -> List[Dict[str, Any]]:
        return [
            {"name": name, "description": meta["description"]}
            for name, meta in self.tools.items()
        ]

    def call(self, tool_name: str, *args, **kwargs) -> Any:
        if tool_name not in self.tools:
            raise KeyError(f"Tool '{tool_name}' not found.")
        return self.tools[tool_name]["instance"].call(*args, **kwargs)


tool_registry = ToolRegistry()
tool_registry.register(insurance_tool)
tool_registry.register(claim_tool)
tool_registry.register(search_tool)
tool_registry.register(code_tool)

# 3) LLM

In [5]:
USE_GEMINI = HAS_GEMINI and bool(os.getenv("GOOGLE_API_KEY"))

if USE_GEMINI:
    print("✅ GOOGLE_API_KEY found via dotenv. Using Gemini.")
    llm_coding = ChatGoogleGenerativeAI(model="gemini-2.0-flash")
    llm_appeal = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

    coding_prompt = PromptTemplate.from_template(
        "Given the visit reason:\n\"{reason}\"\nReturn ONE ICD-10 diagnosis code only."
    )
    appeal_prompt = PromptTemplate.from_template(
        "Write a concise professional appeal letter for a denied medical claim.\n"
        "Patient ID: {patient_id}\n"
        "Reason for denial: {denial_reason}\n"
        "Codes: {codes}\n"
    )

    coding_chain = LLMChain(prompt=coding_prompt, llm=llm_coding)
    appeal_chain = LLMChain(prompt=appeal_prompt, llm=llm_appeal)
else:
    coding_chain = None
    appeal_chain = None
    print("⚠️ GOOGLE_API_KEY not set or langchain-google-genai not installed – using heuristic fallback instead of Gemini.")


def suggest_icd_codes(visit_reason: str) -> List[str]:
    print("🧾 Coding Agent: suggest_icd_codes")
    # Demonstrate using a 'search' tool as context (MCP-style tool usage)
    _search_result = search_tool.call(visit_reason)
    # We don't use the result to change codes (to keep behavior stable),
    # but it is logged in history for context.
    if USE_GEMINI and coding_chain is not None:
        raw = coding_chain.run({"reason": visit_reason})
        code = raw.strip().split()[0].strip(",.;")
        return [code]
    # Heuristic fallback
    text = visit_reason.lower()
    if "diabetes" in text:
        return ["E11.9"]
    if "hypertension" in text or "high blood pressure" in text:
        return ["I10"]
    if "flu" in text or "fever" in text:
        return ["J10.1"]
    return ["R69"]  # Unknown


def generate_appeal_text(patient_id: str, denial_reason: str, codes: List[str]) -> str:
    """LLM-powered (or template) appeal letter."""
    print("📨 Appeal Agent: generate_appeal_text")
    if USE_GEMINI and appeal_chain is not None:
        return appeal_chain.run(
            {
                "patient_id": patient_id,
                "denial_reason": denial_reason,
                "codes": ", ".join(codes or []),
            }
        ).strip()

    # template fallback
    return (
        f"Dear Medical Review Team,\n\n"
        f"I am writing to appeal the denial of claim for patient {patient_id}. "
        f"The claim was denied due to: {denial_reason}. The submitted codes were: {', '.join(codes or [])}.\n"
        f"Based on the documented visit and clinical necessity, I respectfully request reconsideration.\n\n"
        f"Sincerely,\nRCM Agent"
    )


⚠️ GOOGLE_API_KEY not set or langchain-google-genai not installed – using heuristic fallback instead of Gemini.


# 4) LangGraph state + agent nodes

In [6]:
if not HAS_LANGGRAPH:
    raise ImportError("langgraph is required. Install with `pip install langgraph`.")

from typing_extensions import TypedDict


class RCMState(TypedDict, total=False):
    patient_id: str
    insurance_number: str
    visit_reason: str

    insurance_verified: bool
    codes: List[str]
    status: str
    claim_id: Optional[str]
    audit_issues: List[str]
    denial_reason: Optional[str]
    appeal: Optional[str]

    history: List[str]  # for context compaction


def append_history(state: RCMState, message: str) -> None:
    hist = state.get("history") or []
    hist.append(message)
    state["history"] = hist


def compact_context(history: List[str]) -> str:
    """Context compaction – keep last ~500 chars."""
    text = " | ".join(history)
    return text[-500:]


# ----- Agent nodes for LangGraph -----

def intake_agent_node(state: RCMState) -> RCMState:
    print("📥 Intake Agent node")
    verified = insurance_tool.call(state["insurance_number"])
    state["insurance_verified"] = bool(verified)
    append_history(state, f"Intake: verified={verified}")
    return state


def coding_agent_node(state: RCMState) -> RCMState:
    print("🧾 Coding Agent node")
    codes = suggest_icd_codes(state["visit_reason"])
    state["codes"] = codes
    append_history(state, f"Coding: {codes}")
    return state


def claim_agent_node(state: RCMState) -> RCMState:
    print("📄 Claim Agent node (simulating long-running op)")
    result = claim_tool.call(state.get("codes", []), state.get("insurance_verified", False))
    state["status"] = result["status"]
    state["claim_id"] = result.get("claim_id")
    state["denial_reason"] = result.get("reason")
    append_history(state, f"Claim: {result}")
    return state


def audit_agent_node(state: RCMState) -> RCMState:
    print("🔍 Audit Agent node (evaluation)")
    issues: List[str] = []
    if not state.get("insurance_verified"):
        issues.append("Insurance not verified")
    if not state.get("codes"):
        issues.append("Missing codes")
    if state.get("status") != "Submitted":
        issues.append("Claim not successfully submitted")
    # Use the code execution tool to compute a simple issue "score"
    score_expr = str(len(issues))
    score = code_tool.call(score_expr)
    issues.append(f"IssueScore={score}")
    state["audit_issues"] = issues
    append_history(state, f"Audit issues: {issues}")
    return state


def audit_router(state: RCMState) -> str:
    """Loop decision: retry coding or go to appeal."""
    issues = state.get("audit_issues") or []
    if issues and "Missing codes" in issues:
        print("♻️ Audit router: re-running coding due to 'Missing codes'")
        return "retry_coding"
    print("✅ Audit router: proceed to appeal")
    return "ok"


def appeal_agent_node(state: RCMState) -> RCMState:
    print("📨 Appeal Agent node")
    if state.get("status") == "Rejected":
        state["appeal"] = generate_appeal_text(
            state["patient_id"],
            state.get("denial_reason") or "Unknown reason",
            state.get("codes") or [],
        )
    else:
        state["appeal"] = "No appeal needed – claim is submitted."
    append_history(state, "Appeal generated.")
    return state


def memory_store_node(state: RCMState) -> RCMState:
    print("💾 Memory Store node")
    summary = compact_context(state.get("history", []))
    long_term_memory.append(
        {
            "patient_id": state["patient_id"],
            "claim_id": state.get("claim_id"),
            "status": state.get("status"),
            "codes": state.get("codes"),
            "summary": summary,
        }
    )
    return state


# 5)  Build LangGraph workflow

In [7]:
workflow = StateGraph(RCMState)

workflow.add_node("intake", intake_agent_node)
workflow.add_node("coding", coding_agent_node)
workflow.add_node("claim", claim_agent_node)
workflow.add_node("audit", audit_agent_node)
workflow.add_node("appeal", appeal_agent_node)
workflow.add_node("memory", memory_store_node)

workflow.set_entry_point("intake")
workflow.add_edge("intake", "coding")
workflow.add_edge("coding", "claim")
workflow.add_edge("claim", "audit")
workflow.add_conditional_edges(
    "audit",
    audit_router,
    {
        "retry_coding": "coding",  # loop
        "ok": "appeal",
    },
)
workflow.add_edge("appeal", "memory")
workflow.add_edge("memory", END)

graph = workflow.compile()

print("✅ LangGraph workflow compiled.")

✅ LangGraph workflow compiled.


 # 6) A2A pipeline function

In [8]:
def run_rcm_pipeline(state: Dict[str, Any]) -> Dict[str, Any]:
    """
    A2A-style entrypoint: takes a dict state, returns final state.
    This is what we'll expose over FastAPI + Gradio, so other agents
    (or systems) can call this agent as a service.
    """
    # Ensure required keys
    rcm_state: RCMState = RCMState(
        patient_id=state["patient_id"],
        insurance_number=state["insurance_number"],
        visit_reason=state["visit_reason"],
        history=state.get("history", []),
    )
    print("🚀 Invoking RCM agent graph...\n")
    final_state: RCMState = graph.invoke(rcm_state)
    print("\n✅ RCM pipeline finished. Final state:\n", final_state)
    return dict(final_state)

# 7) FastAPI app with A2A JSON endpoint + MCP-style tool API

In [9]:
app = FastAPI(title="Intelligent RCM Agent", version="1.0.0")


class A2ARequest(BaseModel):
    session_id: Optional[str] = None
    patient_id: str
    insurance_number: str
    visit_reason: str


class A2AResponse(BaseModel):
    session_id: str
    state: Dict[str, Any]


class MCPToolInfo(BaseModel):
    name: str
    description: str


class MCPCallRequest(BaseModel):
    tool_name: str
    args: List[Any] = []
    kwargs: Dict[str, Any] = {}


@app.get("/health")
def health():
    return {"status": "ok", "use_gemini": USE_GEMINI}


@app.post("/a2a/rcm", response_model=A2AResponse)
def a2a_rcm(request: A2ARequest):
    """
    A2A-like endpoint:
    - If session_id missing → create new session
    - If session_id provided → resume/update existing state

    This can be called by another agent/service (Agent2Agent).
    """
    if request.session_id:
        state = session_service.get_state(request.session_id) or {}
        state.update(
            {
                "patient_id": request.patient_id,
                "insurance_number": request.insurance_number,
                "visit_reason": request.visit_reason,
            }
        )
        session_id = request.session_id
        print(f"🔄 Resuming session {session_id}")
    else:
        base_state = {
            "patient_id": request.patient_id,
            "insurance_number": request.insurance_number,
            "visit_reason": request.visit_reason,
            "history": [],
        }
        session_id = session_service.create_session(base_state)
        state = base_state
        print(f"🆕 Starting new session {session_id}")

    final_state = run_rcm_pipeline(state)
    session_service.update_state(session_id, final_state)
    return A2AResponse(session_id=session_id, state=final_state)


@app.get("/mcp/tools", response_model=List[MCPToolInfo])
def mcp_list_tools():
    tools = tool_registry.list_tools()
    return [MCPToolInfo(**t) for t in tools]


@app.post("/mcp/call")
def mcp_call_tool(req: MCPCallRequest):
    try:
        result = tool_registry.call(req.tool_name, *req.args, **req.kwargs)
        return {"ok": True, "result": result}
    except Exception as e:
        return {"ok": False, "error": str(e)}

 # 8) Gradio UI for demo purposes

In [10]:
def gradio_run(patient_id: str, insurance_number: str, visit_reason: str):
    """
    Wrapper for UI:
    - Runs the RCM pipeline
    - Returns a human-readable summary, appeal text,
      full state, and last few memory entries.
    """
    state = {
        "patient_id": patient_id,
        "insurance_number": insurance_number,
        "visit_reason": visit_reason,
        "history": [],
    }
    final_state = run_rcm_pipeline(state)

    # High-level summary in Markdown
    summary_lines = [
        f"### 🧾 Claim Summary",
        "",
        f"- **Gemini status:** {'ON ✅' if USE_GEMINI else 'OFF – heuristic fallback'}",
        f"- **Patient ID:** `{final_state.get('patient_id')}`",
        f"- **Insurance Verified:** `{final_state.get('insurance_verified')}`",
        f"- **Suggested Codes:** `{final_state.get('codes')}`",
        f"- **Claim Status:** `{final_state.get('status')}`",
        f"- **Claim ID:** `{final_state.get('claim_id')}`",
        f"- **Audit Issues:** `{final_state.get('audit_issues')}`",
        f"- **History steps:** `{len(final_state.get('history', []))}`",
    ]
    summary_md = "\n".join(summary_lines)

    appeal_text = final_state.get("appeal", "")

    # Show only the last 3 memory entries for readability
    memory_tail = long_term_memory[-3:]

    return summary_md, appeal_text, final_state, memory_tail


# Custom CSS injected via <style> (compatible with older Gradio)
custom_css = """
#rcm-container {
    max-width: 1100px;
    margin-left: auto;
    margin-right: auto;
}

body {
    background: radial-gradient(circle at top, #0f172a 0, #020617 55%);
}

.card {
    border-radius: 14px;
    border: 1px solid #1f2937;
    padding: 16px;
    background: #020617;
    box-shadow: 0 10px 25px rgba(0,0,0,0.45);
}

.card-light {
    border-radius: 14px;
    border: 1px solid #e5e7eb;
    padding: 16px;
    background: #f9fafb;
}

.badge {
    display: inline-block;
    padding: 4px 10px;
    border-radius: 999px;
    font-size: 12px;
    font-weight: 600;
    background: #0f766e;
    color: white;
    margin-right: 6px;
}

.badge-secondary {
    background: #4b5563;
}

h1, h2, h3, h4 {
    color: #e5e7eb !important;
}
"""

with gr.Blocks(title="🧾 Intelligent RCM Agent") as demo:
    # Inject custom CSS manually (works across Gradio versions)
    gr.HTML(f"<style>{custom_css}</style>")

    with gr.Column(elem_id="rcm-container"):
        # Header / Hero
        gr.Markdown(
            """
# 🧾 Intelligent Revenue Cycle Management Agent

Automated workflow for **Intake → Coding → Claim → Audit → Appeal → Memory**.

Use this UI to simulate different patient visits, insurance numbers, and see how the
agents collaborate to process the claim.
            """,
        )

        with gr.Row():
            # Left: Inputs + Examples
            with gr.Column(scale=1):
                gr.Markdown("### Input", elem_classes=["card-light"])

                pid_in = gr.Textbox(
                    label="Patient ID",
                    value="P001",
                    placeholder="e.g. P001"
                )
                ins_in = gr.Textbox(
                    label="Insurance Number",
                    value="AET123456",
                    placeholder="e.g. AET123456 (valid) or XYZ999 (invalid)"
                )
                reason_in = gr.Textbox(
                    label="Visit Reason",
                    value="Patient has type 2 diabetes and hypertension.",
                    lines=3,
                    placeholder="Clinical summary of the visit"
                )

                gr.Markdown("#### Quick examples")
                gr.Examples(
                    examples=[
                        ["P001", "AET123456", "Patient has type 2 diabetes and hypertension."],
                        ["P002", "XYZ999", "Patient has type 2 diabetes and hypertension."],
                        ["P003", "BLU777777", "Patient has flu with high fever and cough."],
                        ["P004", "AET555555", "Patient reports feeling unwell with no clear diagnosis."]
                    ],
                    inputs=[pid_in, ins_in, reason_in],
                    label="Click an example to auto-fill the form"
                )

                run_btn = gr.Button("Run RCM Agent 🚀", variant="primary")

                gr.Markdown(
                    """
> 💡 **Tip:**
> Try a valid insurance number (e.g. `AET123456`) vs an invalid one (e.g. `XYZ999`)
> to see how the audit and appeal agents behave.
                    """
                )

            # Right: Outputs (Summary, Appeal, State, Memory)
            with gr.Column(scale=2):
                with gr.Group(elem_classes=["card"]):
                    gr.Markdown("### Output")

                    with gr.Tabs():
                        with gr.Tab("Summary"):
                            summary_out = gr.Markdown(label="Summary")

                        with gr.Tab("Appeal Letter"):
                            appeal_out = gr.Textbox(
                                label="Generated Appeal (if claim rejected)",
                                lines=14
                            )

                        with gr.Tab("Agent State (JSON)"):
                            state_out = gr.JSON(label="Final Agent State")

                        with gr.Tab("Memory Bank (Last 3 Records)"):
                            memory_out = gr.JSON(label="Last Memory Records")

        run_btn.click(
            fn=gradio_run,
            inputs=[pid_in, ins_in, reason_in],
            outputs=[summary_out, appeal_out, state_out, memory_out],
            show_progress=True,
        )

demo.launch()

print("Use this notebook UI, or run `uvicorn main:app --reload` for the API.")

/usr/local/lib/python3.12/dist-packages/gradio/analytics.py:106: UserWarning: IMPORTANT: You are using gradio version 4.44.0, however version 4.44.1 is available, please upgrade. 
--------
  warnings.warn(


Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Running on public URL: https://ddf44fc96101c98855.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Use this notebook UI, or run `uvicorn main:app --reload` for the API.
